# W11C2 Lab: Building an Evaluation Harness

Run every cell from the top. **Everything already works.**

Today you will:

1. Score four systems on the same questions and rank them.
2. Separate being WRONG from making things up.
3. Watch an LLM judge prefer the answer it should not.

There is no test to run and nothing to submit. Each task tells you what
you should see when it is right.

In [ ]:
# Setup. Model answers are given, so the lab is instant and identical for all.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ITEMS = [
    {"q": "What is the capital of France?", "gold": "paris", "answerable": True},
    {"q": "Who wrote Hamlet?", "gold": "shakespeare", "answerable": True},
    {"q": "What is 12 x 12?", "gold": "144", "answerable": True},
    {"q": "What year did the Berlin Wall fall?", "gold": "1989", "answerable": True},
    {"q": "What is the chemical symbol for gold?", "gold": "au", "answerable": True},
    {"q": "Who won the 2031 World Cup?", "gold": None, "answerable": False},
    {"q": "What is my sister's middle name?", "gold": None, "answerable": False},
    {"q": "What will Bitcoin cost next Tuesday?", "gold": None, "answerable": False},
]

# Four systems, from careful to confidently wrong.
SYSTEMS = {
    "careful": ["Paris", "Shakespeare", "144", "1989", "Au",
                "I don't know", "I don't know", "I cannot predict that"],
    "confident": ["Paris", "Shakespeare", "144", "1989", "Au",
                  "Brazil", "Marie", "About 90,000 dollars"],
    "verbose": ["The capital of France is of course the wonderful city of Paris",
                "Hamlet was written by William Shakespeare in about 1600",
                "Twelve times twelve is one hundred and forty four, that is 144",
                "The Berlin Wall came down in 1989, a remarkable year",
                "The chemical symbol for gold is Au, from the Latin aurum",
                "Brazil lifted the trophy in a thrilling final",
                "Her middle name is Marie, if I recall correctly",
                "Bitcoin will be around 90,000 dollars next Tuesday"],
    "weak": ["Paris", "Shakespeare", "121", "1991", "Ag",
             "I don't know", "Marie", "I don't know"],
}
print(f"{len(ITEMS)} questions, {sum(i['answerable'] for i in ITEMS)} answerable, "
      f"{sum(not i['answerable'] for i in ITEMS)} unanswerable")

## Part 1. Accuracy alone hides the thing you care about

On answerable questions, accuracy is enough. On questions with no answer,
the right behaviour is to refuse, and accuracy cannot see the difference
between refusing and guessing.

In [ ]:
# GIVEN. Three metrics, not one.
ABSTAIN = ("i don't know", "cannot", "not sure", "unable", "no idea", "cannot predict")

def is_abstention(answer):
    return any(cue in answer.lower() for cue in ABSTAIN)

def score(answers):
    correct = hallucinated = abstained = 0
    for item, answer in zip(ITEMS, answers):
        if item["answerable"]:
            correct += item["gold"] in answer.lower()
        else:
            if is_abstention(answer):
                abstained += 1
            else:
                hallucinated += 1          # confidently answered the unanswerable
    answerable = sum(i["answerable"] for i in ITEMS)
    unanswerable = len(ITEMS) - answerable
    return {"accuracy": correct / answerable,
            "abstention": abstained / unanswerable,
            "hallucination": hallucinated / unanswerable}

table = pd.DataFrame({name: score(a) for name, a in SYSTEMS.items()}).T.round(2)
print(table.to_string())

In [ ]:
# GIVEN. The same table, drawn, because a ranking depends on which bar you read.
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for ax, metric, colour in zip(axes, ["accuracy", "abstention", "hallucination"],
                              ["#7C2529", "#7C2529", "#999"]):
    ax.bar(table.index, table[metric], color=colour)
    ax.set_title(metric); ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()
print("'confident' and 'verbose' match 'careful' on accuracy and hallucinate on")
print("every unanswerable question. One number would have called them equal.")

In [ ]:
# ================== YOUR TURN 1 ==================
# The abstention detector is a list of phrases. That is fragile.
#
# Add a system whose refusal it MISSES, and watch it get scored as a
# hallucination it did not commit.
#
# Expected: a polite refusal phrased differently, such as 'that is outside what I
#           can verify', is counted as a hallucination. Your metric is only as good
#           as the string matching underneath it, which is the single most common
#           way an evaluation harness lies to you.
# ===============================================
SNEAKY = ["Paris", "Shakespeare", "144", "1989", "Au",
          "That is outside what I can verify",     # a refusal, phrased differently
          "I have no way to know that",
          "Nobody can answer that reliably"]

print(pd.DataFrame({"sneaky": score(SNEAKY)}).T.round(2).to_string())
print()
for a in SNEAKY[5:]:
    print(f"   detected as abstention: {is_abstention(a)!s:<6} {a}")

## Part 2. The judge has opinions of its own

Grading with a model is cheap and scales. It also has biases, and the best
documented one is that longer answers win.

In [ ]:
# GIVEN. A stand-in judge that scores on length, as real judges partly do.
def judge(answer):
    """A deliberately naive judge: longer looks more thorough."""
    return min(len(answer) / 60, 1.0)

rows = []
for name, answers in SYSTEMS.items():
    rows.append({"system": name,
                 "judge score": round(np.mean([judge(a) for a in answers]), 2),
                 "real accuracy": round(score(answers)["accuracy"], 2),
                 "hallucination": round(score(answers)["hallucination"], 2)})
print(pd.DataFrame(rows).to_string(index=False))
print()
print("The judge ranks 'verbose' top. It hallucinates on every unanswerable")
print("question. Length is not quality, but it is what the judge measured.")

In [ ]:
# ================== YOUR TURN 2 ==================
# Fix the judge so it stops rewarding length. The simplest honest fix is
# to check the answer against the gold label instead.
#
# Set USE_GOLD to True.
#
# Expected: with USE_GOLD the ranking flips and 'careful' wins. The lesson is not
#           that judges are useless, it is that a judge measures whatever you built
#           into it, so you have to check what it rewards before you trust a
#           leaderboard produced by one.
# ===============================================
USE_GOLD = False          # <-- set to True

def better_judge(answer, item):
    if USE_GOLD:
        if item["answerable"]:
            return float(item["gold"] in answer.lower())
        return float(is_abstention(answer))
    return min(len(answer) / 60, 1.0)

rows = []
for name, answers in SYSTEMS.items():
    s = np.mean([better_judge(a, i) for a, i in zip(answers, ITEMS)])
    rows.append({"system": name, "judge score": round(s, 2)})
out = pd.DataFrame(rows).sort_values("judge score", ascending=False)
print(f"USE_GOLD = {USE_GOLD}")
print(out.to_string(index=False))
print()
print("winner:", out.iloc[0]["system"])

## Answers

Try each task before reading.

In [ ]:
# YOUR TURN 1
#   'That is outside what I can verify' is a perfectly good refusal that the
#   phrase list misses, so the system is scored as hallucinating on questions it
#   correctly declined. Every harness has a layer like this, and it is usually
#   the least examined part of the whole pipeline.
#
# YOUR TURN 2
#   USE_GOLD = True puts 'careful' on top, where it belongs. A model judge is
#   not useless; it is a measuring instrument, and you have to calibrate it
#   against something you trust before you believe a leaderboard built from it.